# Interpretación de modelos ARMA(p,q)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

total_time = 100
t = np.arange(total_time)

alpha = 0.1

mu, sigma = 0.1, 0.1 # mean and standard deviation
# mu no nulo para compensar la salida abierta 
rng = np.random.default_rng(42)
s = rng.normal(mu, sigma, total_time)

changes = 100
sub_s = s[:changes]

w = np.repeat(sub_s, total_time//changes)

xh = np.exp(-alpha*t)

xp = xh*np.cumsum(w*np.exp(alpha*t))

x = xh+xp

df = pd.DataFrame({'xh': xh, 'w':w, 'x':x})

df.plot(subplots=True)

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf
plot_acf(x, lags=20, bartlett_confint=False)
plt.xlabel('Lag')

In [ ]:
from statsmodels.tsa.stattools import pacf
from statsmodels.graphics.tsaplots import plot_pacf

plot_pacf(x, lags=20)
plt.xlabel('Lag')

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
from itertools import product
from tqdm.notebook import tqdm # make your loops show a smart progress meter

max_order = 5 
ps = qs = range(max_order)
aic_final_df = pd.DataFrame({'p':[], 'q':[], 'AIC':[]})

for p,q in tqdm(product(ps,qs)): # Cartesian product of the input iterables.
    try:
        model = ARIMA(x, order=(p, 0, q), trend='c')
    except:
        continue
    res = model.fit()
    aic_df = pd.DataFrame({'p':[p], 'q':[q], 'AIC':[res.aic]})
    aic_final_df = pd.concat([aic_final_df, aic_df])

            

aic_final_df.sort_values(by='AIC')

In [ ]:
model = ARIMA(x, order=(1, 0, 0), trend='c')
res = model.fit()
print('Sumary: ', res.summary())
print('Params: ', model.params_complete)

In [ ]:
# Predicted values with fitted model
predicted_series = res.get_prediction().predicted_mean


# Plot original and simulated data
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(x,label='Original Data', color='k')
ax.plot(predicted_series, label='Predicted Data', color='C1', alpha=0.5)
ax.set_title("Original and Predicted Time Series from fitted ARIMA(1,0,0)")
plt.legend()

In [ ]:
x[0]

In [ ]:
# Plot original and simulated data
fig, ax = plt.subplots(figsize=(10, 5))

num_solution = np.ones(total_time)*x[0]
for i in range(total_time-1):
    num_solution[i+1] = num_solution[i] / 1.1 + w[i+1] / 1.1
    
num2_solution = num_solution.copy()
for i in range(total_time-2):
    num2_solution[i+2] = (2 * num2_solution[i+1] - 0.5*num2_solution[i] + w[i+2] )/ 1.6

ax.plot(x,label='Original Data', color='k')
ax.plot(num_solution, label='Num.solution 1', color='C1', alpha=0.5)
ax.plot(num2_solution, label='Num.solution 2', color='C2', alpha=0.5)
ax.set_title("Original and Numerical Solutions Time Series")
plt.legend()

In [ ]:
model = ARIMA(x, order=(1, 0, 1), trend='c')
res = model.fit()
print('Sumary: ', res.summary())
print('Params: ', model.params_complete)